<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_01_logistic_regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_01 - T2 SEQ2ONE - Baseline**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [56]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-03-24 05:11:31,503 | INFO | Environment initialized


## **2. Acceso a drive**

In [57]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-03-24 05:11:32,171 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [58]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

# Tamaños de ventana
WINDOW_SIZES = [30, 60, 90, 120, 180]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-03-24 05:11:32,179 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-03-24 05:11:32,180 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-03-24 05:11:32,181 | INFO | Configuración de experimento cargada
2026-03-24 05:11:32,182 | INFO | Targets: ['t2_dir_thr_90', 't2_dir_thr_120']
2026-03-24 05:11:32,184 | INFO | Window sizes: [30, 60, 90, 120, 180]


In [59]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:5]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[60]["t2_dir_thr_90"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-03-24 05:11:32,195 | INFO | Paths construidos (windows + scaler compartido T2)
2026-03-24 05:11:32,204 | INFO | Windows OK      : 30
2026-03-24 05:11:32,205 | INFO | Windows missing : 0
2026-03-24 05:11:32,206 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-03-24 05:11:32,207 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [60]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [61]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [62]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_dir_thr_90'
        - 't2_dir_thr_120'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }


### **4.4. Creación de bundles T2**

In [63]:
# --------------------------------------------------
# Crea bundles T2 para un window_size dado
# --------------------------------------------------
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundle_t2_90  : dict
    bundle_t2_120 : dict
    """

    if len(targets) != 2:
        raise ValueError(
            f"Se esperaban exactamente 2 targets T2. Recibido: {targets}"
        )

    bundle_t2_90 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[0],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    bundle_t2_120 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[1],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    # --------------------------------------------------
    # Verificación rápida
    # --------------------------------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    print(f"\nTARGET: {targets[0]}")
    print("Train :", bundle_t2_90["train"]["X"].shape, bundle_t2_90["train"]["y"].shape)
    print("Valid :", bundle_t2_90["valid"]["X"].shape, bundle_t2_90["valid"]["y"].shape)
    print("Test  :", bundle_t2_90["test"]["X"].shape,  bundle_t2_90["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_90["scaler"]).__name__)

    print(f"\nTARGET: {targets[1]}")
    print("Train :", bundle_t2_120["train"]["X"].shape, bundle_t2_120["train"]["y"].shape)
    print("Valid :", bundle_t2_120["valid"]["X"].shape, bundle_t2_120["valid"]["y"].shape)
    print("Test  :", bundle_t2_120["test"]["X"].shape,  bundle_t2_120["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_120["scaler"]).__name__)

    return bundle_t2_90, bundle_t2_120

In [64]:
#bundle_t2_90, bundle_t2_120 = create_bundles(window_size=60)

Como acceder a las ventanas X e y:

```python
X_train_90 = bundle_t2_90["train"]["X"]
y_train_90 = bundle_t2_90["train"]["y"]

X_valid_120 = bundle_t2_120["valid"]["X"]
y_valid_120 = bundle_t2_120["valid"]["y"]

scaler = bundle_t2_90["scaler"]
```




### **4.5. Preparación de inputs según el tipo de modelo**

In [65]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

In [66]:
from __future__ import annotations

from typing import Any, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)

    Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1). Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )

In [67]:
# ============================================================
# 2) Sanity check principal (seq2one clasificación T2)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como d_flat esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado).
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len), permite tomar y[:, -1].
        Por defecto False.
    """

    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Normalización de y
    # --------------------------------------------------
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        y = y[:, -1]

    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Inferir modo y dimensiones de X
    # --------------------------------------------------
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # --------------------------------------------------
    # Validación básica de n_samples
    # --------------------------------------------------
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # --------------------------------------------------
    # Validación de shapes según modo
    # --------------------------------------------------
    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    # --------------------------------------------------
    # Diagnóstico de clases
    # --------------------------------------------------
    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    # --------------------------------------------------
    # Salida informativa
    # --------------------------------------------------
    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info

In [68]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_dir_thr_90",
      "horizon": 90,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # --------------------------------------------------
    # Setear esperados desde TRAIN si no se dieron
    # --------------------------------------------------
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    # --------------------------------------------------
    # Ejecutar checks
    # --------------------------------------------------
    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_targets_seq2one(
    bundle_t2_90: Dict[str, Any],
    bundle_t2_120: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para ambos targets T2.
    """
    out_t2_90 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_90,
        tag="t2_90",
        verbose=verbose,
    )

    out_t2_120 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_120,
        tag="t2_120",
        verbose=verbose,
    )

    return {
        "t2_dir_thr_90": out_t2_90,
        "t2_dir_thr_120": out_t2_120,
    }

In [69]:
#sanity_outputs = run_sanity_checks_all_targets_seq2one(
#    bundle_t2_90,
#    bundle_t2_120,
#    verbose=True,
#)

In [70]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **6. Métricas de clasificación T2**

In [71]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-03-24 05:11:32,325 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [72]:
# ================================
# Carga de métricas (si existen)
# ================================

def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.
    """

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(f"No existen métricas previas para: {name}")
    return pd.DataFrame()

In [73]:
# ================================
# Guardado de métricas
# ================================

def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path

Ejemplo de uso:

```python
df_metrics = load_classification_metrics_if_exists(name="lstm_valid")

df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)

save_classification_metrics(df_metrics, name="lstm_valid")
```



## **8. Gestión de dispositivo y memoria**

In [74]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [75]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-03-24 05:11:32,354 | INFO | Seeds fijadas en 42


# **DEFINICIÓN DE MODELO**

## Estructura general a replicar en todos los modelos


### 1. Función de predicción / entrenamiento por bundle

**Objetivo:** trabajar con una sola combinación experimental.

Entrada:

* un `bundle`

Responsabilidad:

* extraer `train/valid/test`
* entrenar el modelo con TRAIN
* generar predicciones en VALID y TEST

Salida:

* diccionario con:

  * modelo entrenado
  * `y_pred_valid`
  * `y_pred_test`

#### Forma conceptual

```python
run_model_for_bundle(bundle) -> {
    "model": ...,
    "y_pred_valid": ...,
    "y_pred_test": ...,
}
```

---

### 2. Función de evaluación por bundle o por lista de bundles

**Objetivo:** convertir predicciones en métricas.

Entrada:

* uno o varios bundles
* split a evaluar
* nombre del modelo
* parámetros de evaluación

Responsabilidad:

* llamar a la función del paso 1
* tomar `y_true` del split correspondiente
* calcular métricas
* convertirlas a DataFrame

Salida:

* DataFrame con una fila por bundle evaluado

#### Forma conceptual

```python
eval_model_bundles(
    bundles,
    split="valid",
    model_name="...",
) -> pd.DataFrame
```

---

### 3. Función orquestadora por `window_size`

**Objetivo:** correr el experimento completo para una sola ventana.

Entrada:

* `window_size`

Responsabilidad:

* crear bundles para esa ventana
* correr ambos targets
* evaluar VALID y TEST
* concatenar todas las filas
* ordenar resultados
* liberar memoria

Salida:

* DataFrame final para esa ventana

#### Forma conceptual

```python
run_model(window_size=60) -> pd.DataFrame
```

---

### 4. Función incremental sobre múltiples `window_size`

**Objetivo:** correr el experimento completo sobre varias ventanas y mantener histórico.

Entrada:

* lista de `window_sizes`
* nombre del experimento/modelo

Responsabilidad:

* cargar histórico si existe
* verificar qué ventanas ya están completas
* hacer skip si corresponde
* correr `run_model(window_size=L)` para ventanas faltantes
* concatenar al histórico
* guardar en parquet
* devolver histórico final ordenado

Salida:

* DataFrame histórico consolidado

#### Forma conceptual

```python
run_model_incremental(window_sizes=[30, 60, 90, ...]) -> pd.DataFrame
```

---

### Patrón completo del pipeline por modelo

**Paso 1**: **Bundle → predicciones**

**Paso 2**: **Predicciones → métricas**

**Paso 3**: **Métricas → DataFrame**

**Paso 4**: **Una ventana → experimento completo**

**Paso 5**: **Múltiples ventanas → histórico incremental**

---

# Qué debe mantenerse igual en todos los modelos

## A. La interfaz del bundle

Siempre:

```python
bundle["train"]["X"]
bundle["train"]["y"]
bundle["valid"]["X"]
bundle["valid"]["y"]
bundle["test"]["X"]
bundle["test"]["y"]
bundle["scaler"]
bundle["target"]
bundle["window_size"]
bundle["horizon"]
```

---

## B. La separación de responsabilidades

No mezclar en una sola función:

* entrenamiento
* métricas
* guardado
* iteración por ventanas

Cada capa debe hacer solo una cosa.

---

## C. La salida tabular final

Todos los modelos deben terminar produciendo un DataFrame con la misma estructura de columnas, para poder comparar resultados.

---

## D. La persistencia incremental

Todos los modelos deben tener la capacidad de:

* cargar histórico
* saltear ventanas ya procesadas
* guardar resultados nuevos

---

# Qué cambia entre modelos

Lo único que cambia de un modelo a otro es:

## 1. La función del paso 1

Ejemplos:

* Logistic Regression
* Random Forest
* XGBoost
* LightGBM
* LSTM
* GRU
* Transformer

Es decir, cambia:

* cómo se prepara `X`
* cómo se entrena
* cómo se predice

---

## 2. El `input_mode`

* tabulares → `"2d_flat"`
* secuenciales → `"3d"`

---

## 3. Algunos hiperparámetros

Pero la estructura general no cambia.

---

# Plantilla general por modelo

## 1. `run_<model>_for_bundle(...)`

Predice VALID y TEST para un bundle.

## 2. `eval_<model>_bundles(...)`

Calcula métricas y devuelve tabla.

## 3. `run_<model>(window_size=...)`

Corre ambos targets para una ventana.

## 4. `run_<model>_incremental(window_sizes=...)`

Corre múltiples ventanas con persistencia incremental.

---

# Conclusión

La estructura estándar a replicar en todos los modelos es esta:

### 1. Función unitaria por bundle

entrena y predice

### 2. Función evaluadora

calcula métricas y arma DataFrame

### 3. Función por ventana

corre ambos targets para un `window_size`

### 4. Función incremental

recorre varias ventanas, evita repeticiones y guarda histórico

Esa es la plantilla que yo usaría exactamente para Logistic Regression y luego copiaría para los demás modelos.


## **10. Modelo baseline: Logistic Regression**

In [77]:
# ================================
# Modelo baseline: Logistic Regression
# ================================

from sklearn.linear_model import LogisticRegression


def train_logistic_regression_baseline(
    bundle: dict,
    *,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    input_mode: str = "2d_flat",
) -> dict:
    """
    Entrena un baseline de Logistic Regression para clasificación T2
    y evalúa el desempeño sobre VALID.

    Parámetros
    ----------
    bundle : dict
        Bundle con estructura:
        {
            "target": ...,
            "window_size": ...,
            "train": {"X": ..., "y": ...},
            "valid": {"X": ..., "y": ...},
            "test":  {"X": ..., "y": ...},
            "scaler": ...
        }

    model_name : str
        Nombre del modelo para reporting.

    max_iter : int
        Número máximo de iteraciones para LogisticRegression.

    random_state : int
        Semilla interna del modelo.

    input_mode : str
        Modo de preparación de X:
        - "2d_flat" para modelos tabulares
        - "3d" no aplica aquí, pero se deja por consistencia

    Retorna
    -------
    dict
        Diccionario con:
        - modelo entrenado
        - métricas VALID
        - DataFrame de métricas VALID
        - predicciones VALID
    """

    # --------------------------------------------------
    # 1) Metadata del experimento
    # --------------------------------------------------
    target = bundle["target"]
    window_size = bundle["window_size"]

    logger.info("=" * 80)
    logger.info(f"[START] {model_name}")
    logger.info(f"Target      : {target}")
    logger.info(f"Window size : {window_size}")
    logger.info(f"Input mode  : {input_mode}")
    logger.info("=" * 80)

    # --------------------------------------------------
    # 2) Extraer TRAIN y VALID desde el bundle
    # --------------------------------------------------
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    logger.info("[1/6] Datos extraídos desde el bundle")
    logger.info(f"X_train raw shape: {X_train.shape}")
    logger.info(f"y_train raw shape: {y_train.shape}")
    logger.info(f"X_valid raw shape: {X_valid.shape}")
    logger.info(f"y_valid raw shape: {y_valid.shape}")

    # --------------------------------------------------
    # 3) Preparar inputs para modelo tabular
    #    LogisticRegression espera X 2D:
    #    (n_samples, seq_len * n_features)
    # --------------------------------------------------
    logger.info("[2/6] Preparando inputs para Logistic Regression")

    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    logger.info(f"X_train model shape: {X_train_model.shape}")
    logger.info(f"X_valid model shape: {X_valid_model.shape}")

    # --------------------------------------------------
    # 4) Definir el modelo
    # --------------------------------------------------
    logger.info("[3/6] Inicializando modelo")

    model = LogisticRegression(
        max_iter=max_iter,
        random_state=random_state,
        multi_class="multinomial",
        class_weight="balanced",
        n_jobs=-1,
    )

    logger.info(
        f"Modelo creado | max_iter={max_iter} | random_state={random_state} | "
        f"class_weight='balanced' | multi_class='multinomial'"
    )

    # --------------------------------------------------
    # 5) Entrenamiento
    # --------------------------------------------------
    logger.info("[4/6] Entrenando modelo...")

    model.fit(X_train_model, y_train)

    logger.info("[OK] Entrenamiento finalizado")

    # --------------------------------------------------
    # 6) Predicción sobre VALID
    # --------------------------------------------------
    logger.info("[5/6] Generando predicciones sobre VALID")

    y_pred_valid = model.predict(X_valid_model)

    logger.info(f"[OK] Predicciones generadas | y_pred_valid shape: {y_pred_valid.shape}")

    # --------------------------------------------------
    # 7) Cálculo de métricas
    # --------------------------------------------------
    logger.info("[6/6] Calculando métricas de clasificación sobre VALID")

    metrics_valid = compute_classification_metrics(
        y_true=y_valid,
        y_pred=y_pred_valid,
        model_name=model_name,
        split="valid",
        target=target,
        labels=[-1, 0, 1],
    )

    metrics_valid_df = metrics_to_df(
        metrics_valid,
        model=model_name,
        split="valid",
        window_size=window_size,
        target=target,
    )

    logger.info("[OK] Métricas calculadas")
    logger.info(
        f"balanced_accuracy={metrics_valid['metrics']['balanced_accuracy']:.6f} | "
        f"f1_macro={metrics_valid['metrics']['f1_macro']:.6f} | "
        f"accuracy={metrics_valid['metrics']['accuracy']:.6f}"
    )

    logger.info(f"[END] {model_name} | target={target} | window_size={window_size}")
    logger.info("=" * 80)

    # --------------------------------------------------
    # 8) Retorno
    # --------------------------------------------------
    return {
        "model": model,
        "metrics_valid": metrics_valid,
        "metrics_valid_df": metrics_valid_df,
        "y_pred_valid": y_pred_valid,
    }

In [78]:
bundle_t2_90, bundle_t2_120 = create_bundles(window_size=60)

result_t2_90 = train_logistic_regression_baseline(bundle_t2_90)
print_classification_report_block(result_t2_90["metrics_valid"])
display(result_t2_90["metrics_valid_df"])

2026-03-24 05:19:39,416 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-03-24 05:19:39,416 | INFO | X shape: (409512, 60, 5) | y shape: (409512,)
2026-03-24 05:19:39,690 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-03-24 05:19:39,691 | INFO | X shape: (87688, 60, 5) | y shape: (87688,)
2026-03-24 05:19:39,969 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-03-24 05:19:39,970 | INFO | X shape: (88140, 60, 5) | y shape: (88140,)
2026-03-24 05:19:39,975 | INFO | Scaler cargado: scaler_t2.pkl
2026-03-24 05:19:39,976 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=60 | train=(409512, 60, 5) | valid=(87688, 60, 5) | test=(88140, 60, 5)
2026-03-24 05:19:41,348 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-03-24 05:19:41,349 | INFO | X shape: (409512, 60, 5) | y shape: (409512,)
2026-03-24 05:19:41,632 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-03-24 05:19:41,633 | INFO | X shape: (87688, 60, 5) | y shape: (87688,)
2026-03-24 05:19:41,913 |


WINDOW_SIZE: 60

TARGET: t2_dir_thr_90
Train : (409512, 60, 5) (409512,)
Valid : (87688, 60, 5) (87688,)
Test  : (88140, 60, 5) (88140,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (409512, 60, 5) (409512,)
Valid : (87688, 60, 5) (87688,)
Test  : (88140, 60, 5) (88140,)
Scaler: StandardScaler


2026-03-24 05:23:55,906 | INFO | [OK] Entrenamiento finalizado
2026-03-24 05:23:55,907 | INFO | [5/6] Generando predicciones sobre VALID
2026-03-24 05:23:56,012 | INFO | [OK] Predicciones generadas | y_pred_valid shape: (87688,)
2026-03-24 05:23:56,013 | INFO | [6/6] Calculando métricas de clasificación sobre VALID
2026-03-24 05:23:56,334 | INFO | [OK] Métricas calculadas
2026-03-24 05:23:56,335 | INFO | balanced_accuracy=0.416490 | f1_macro=0.420733 | accuracy=0.635150
2026-03-24 05:23:56,335 | INFO | [END] logistic_regression | target=t2_dir_thr_90 | window_size=60
2026-03-24 05:23:56,336 | INFO | ================================================================================


CLASSIFICATION REPORT | model=logistic_regression | split=valid | target=t2_dir_thr_90
n_samples               : 87688
balanced_accuracy       : 0.416490
f1_macro                : 0.420733
f1_weighted             : 0.619749
accuracy                : 0.635150
precision_macro         : 0.429252
recall_macro            : 0.416490
--------------------------------------------------------------------------------------
balanced_accuracy_naive : 0.333333
f1_macro_naive          : 0.274034
f1_weighted_naive       : 0.573778
bal_acc_gain_vs_naive   : 0.083157
f1_macro_gain_vs_naive  : 0.146699


,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive
0,logistic_regression,valid,60,t2_dir_thr_90,87688,0.41649,0.420733,0.619749,0.63515,0.429252,0.41649,0.333333,0.083157


In [ ]:
def run_naive_for_bundle_seq2one(bundle, baseline="zero"):
    """
    Ejecuta un modelo Naive para un bundle (train/valid/test).

    baseline:
      - "zero": predice siempre 0
      - "mean": predice el promedio de y_train
    """
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    X_test  = bundle["test"]["X"]

    if baseline == "zero":
        y_pred_valid = predict_naive_zero(X_valid)
        y_pred_test  = predict_naive_zero(X_test)

    elif baseline == "mean":
        y_pred_valid = predict_naive_mean(X_valid, y_train)
        y_pred_test  = predict_naive_mean(X_test, y_train)

    else:
        raise ValueError("baseline debe ser 'zero' o 'mean'")

    return {
        "y_pred_valid": y_pred_valid,
        "y_pred_test": y_pred_test,
    }


In [ ]:
def preds_zero (bundle):
    # Ejecutar Naive ZERO
    preds_zero = run_naive_for_bundle_seq2one(bundle, baseline="zero")

    # Métricas en VALID
    metrics_valid_zero = compute_seq2one_metrics(
        bundle["valid"]["y"],
        preds_zero["y_pred_valid"],
        compute_r2=True
    )

    # Métricas en TEST
    metrics_test_zero = compute_seq2one_metrics(
        bundle["test"]["y"],
        preds_zero["y_pred_test"],
        compute_r2=True
    )

    # Ejecutar Naive MEAN
    preds_mean = run_naive_for_bundle_seq2one(bundle, baseline="mean")

    metrics_valid_mean = compute_seq2one_metrics(
        bundle["valid"]["y"],
        preds_mean["y_pred_valid"],
        compute_r2=True
    )

    metrics_test_mean = compute_seq2one_metrics(
        bundle["test"]["y"],
        preds_mean["y_pred_test"],
        compute_r2=True
    )

    return metrics_valid_zero, metrics_test_zero, metrics_valid_mean, metrics_test_mean

## **11. Métricas de desempeño**

### **11.1. Naive Zero**

In [ ]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd

def eval_naive_seq2one_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    baseline: str = "zero",
    split: str = "valid",
    model_name: str | None = None,
    compute_r2: bool = True,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa un baseline naive (seq2one) para uno o varios bundles y retorna una tabla.

    Requisitos del bundle:
      bundle["train"/"valid"/"test"]["y"]
      bundle["horizon"] (opcional, pero recomendado)
      bundle["target"]  (opcional)

    Usa:
      run_naive_for_bundle_seq2one(bundle, baseline=...)
      compute_seq2one_metrics(y_true, y_pred, ...)
      metrics_to_df(metrics, model=..., split=..., horizon=...)
    """
    # Normalizar a lista
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    if split not in ("train", "valid", "test"):
        raise ValueError("split debe ser 'train', 'valid' o 'test'")

    rows = []

    for bundle in bundles_list:

        if verbose:
            print(
                f"  -> L{bundle['window_size']} | "
                f"target={bundle['target']} | "
                f"split={split} | "
                f"baseline={baseline}"
            )

        # Horizon / target (si existen)
        horizon = int(bundle.get("horizon", -1))
        target = bundle.get("target", None)

        # Correr naive
        preds = run_naive_for_bundle_seq2one(bundle, baseline=baseline)

        # Tomar y_true / y_pred para el split solicitado
        y_true = bundle[split]["y"]
        y_pred_key = f"y_pred_{split}"
        if y_pred_key not in preds:
            raise KeyError(f"No existe '{y_pred_key}' en la salida de run_naive_for_bundle_seq2one")

        y_pred = preds[y_pred_key]

        # Métricas
        metrics = compute_seq2one_metrics(y_true, y_pred, compute_r2=compute_r2)

        # Nombre del modelo (auto si no lo pasas)
        if model_name is not None:
            mname = model_name
        else:
            mname = f"naive_{baseline}" + (f"_{target}" if target else "")

        # A DataFrame (usa horizon si existe; si no, intenta inferirlo desde target)
        if horizon == -1 and isinstance(target, str) and "_" in target:
            try:
                horizon = int(target.split("_")[-1])
            except Exception:
                horizon = -1

        df_row = metrics_to_df(
            metrics, model=mname,
            split=split, horizon=horizon,
            window_size=bundle["window_size"],
            target=bundle["target"],
            )
        rows.append(df_row)

    return pd.concat(rows, ignore_index=True)


In [ ]:
import gc
import pandas as pd

def run_naive(window_size: int, *, verbose: bool = True) -> pd.DataFrame:
    size = int(window_size)

    bundle_delta_60 = bundle_delta_90 = None
    bundle_ret_60 = bundle_ret_90 = None
    bundles_delta = bundles_ret = None

    df_out = None  # df final a retornar

    try:
        if verbose:
            print("\n" + "=" * 80)
            print(f"NAIVE BASELINES | SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)

        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['delta_60', 'delta_90']")
        bundle_delta_60, bundle_delta_90 = create_bundles(
            window_size=size,
            targets=["delta_60", "delta_90"],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
        )

        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['ret_60', 'ret_90']")
        bundle_ret_60, bundle_ret_90 = create_bundles(
            window_size=size,
            targets=["ret_60", "ret_90"],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
        )

        bundles_delta = [bundle_delta_60, bundle_delta_90]
        bundles_ret   = [bundle_ret_60, bundle_ret_90]

        dfs = []

        for split in ["valid", "test"]:
            if verbose:
                print(f"\n[EVAL] L{size} | split={split} | baseline=zero | DELTA")
            dfs.append(
                eval_naive_seq2one_bundles(
                    bundles_delta, baseline="zero", split=split,
                    model_name="naive_zero", compute_r2=True, verbose=True,
                )
            )

            if verbose:
                print(f"[EVAL] L{size} | split={split} | baseline=mean | DELTA")
            dfs.append(
                eval_naive_seq2one_bundles(
                    bundles_delta, baseline="mean", split=split,
                    model_name="naive_mean", compute_r2=True, verbose=True,
                )
            )

            if verbose:
                print(f"[EVAL] L{size} | split={split} | baseline=zero | RET")
            dfs.append(
                eval_naive_seq2one_bundles(
                    bundles_ret, baseline="zero", split=split,
                    model_name="naive_zero", compute_r2=True, verbose=True,
                )
            )

            if verbose:
                print(f"[EVAL] L{size} | split={split} | baseline=mean | RET")
            dfs.append(
                eval_naive_seq2one_bundles(
                    bundles_ret, baseline="mean", split=split,
                    model_name="naive_mean", compute_r2=True, verbose=True,
                )
            )

        df_out = (
            pd.concat(dfs, ignore_index=True)
            .sort_values(["window_size", "target", "split", "horizon_min", "model"])
            .reset_index(drop=True)
        )

        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[["window_size", "split", "target", "model", "horizon_min"]]
                .drop_duplicates()
                .sort_values(["split", "target", "model", "horizon_min"])
                .to_string(index=False)
            )

        return df_out

    finally:
        # liberar memoria grande (bundles con X/y)
        del bundle_delta_60, bundle_delta_90, bundle_ret_60, bundle_ret_90
        del bundles_delta, bundles_ret
        gc.collect()

In [ ]:
from pathlib import Path
import pandas as pd

def run_naive_incremental(
    *,
    window_sizes: list[int],
    name: str = "naive",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Ejecuta Naive de forma incremental para múltiples window_sizes.
    - Carga histórico desde Drive (si existe)
    - Hace SKIP si el L ya está completo (targets + splits esperados)
    - Agrega resultados nuevos al histórico
    - Guarda usando save_seq2one_metrics(df_hist, name=name)
    """

    metrics_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")
    metrics_path = metrics_dir / f"seq2one_{name}_metrics.parquet"

    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    expected_targets = {"delta_60", "delta_90", "ret_60", "ret_90"}
    expected_splits  = {"valid", "test"}
    expected_models  = {"naive_zero", "naive_mean"}  # lo que realmente produce run_naive()

    for L in window_sizes:
        L = int(L)

        # ---- Skip robusto por L ----
        if not df_hist.empty:
            dfL = df_hist[df_hist["window_size"] == L]

            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits  = set(dfL["split"].unique()) if not dfL.empty else set()
            done_models  = set(dfL["model"].unique()) if not dfL.empty else set()

            is_complete = (
                expected_targets.issubset(done_targets)
                and expected_splits.issubset(done_splits)
                and expected_models.issubset(done_models)
            )

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name} L={L} ya existe completo en Drive")
                continue

        # Ejecutar Naive para este L (debe devolver valid+test)
        df_L = run_naive(L)

        # (Opcional) etiqueta común sin pisar "model"
        df_L["family"] = name

        # Actualizar histórico
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # Guardar
        save_seq2one_metrics(df_hist, name=name)

    return df_hist.sort_values(
        ["window_size", "target", "split", "horizon_min", "model"]
    ).reset_index(drop=True)

In [ ]:
df_naive_all_sizes = run_naive_incremental(window_sizes=window_sizes, name="naive", verbose=True)
df_naive_all_sizes


NAIVE BASELINES | SEQ2ONE | WINDOW_SIZE=L30

[BUILD] L30 | targets = ['delta_60', 'delta_90']
H60 Train: (463872, 30, 36) (463872,)
H60 Valid: (99328, 30, 36) (99328,)
H60 Test : (99840, 30, 36) (99840,)
H90 Train: (463872, 30, 36) (463872,)
H90 Valid: (99328, 30, 36) (99328,)
H90 Test : (99840, 30, 36) (99840,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler

[BUILD] L30 | targets = ['ret_60', 'ret_90']
H60 Train: (463872, 30, 36) (463872,)
H60 Valid: (99328, 30, 36) (99328,)
H60 Test : (99840, 30, 36) (99840,)
H90 Train: (463872, 30, 36) (463872,)
H90 Valid: (99328, 30, 36) (99328,)
H90 Test : (99840, 30, 36) (99840,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler

[EVAL] L30 | split=valid | baseline=zero | DELTA
  -> L30 | target=delta_60 | split=valid | baseline=zero
  -> L30 | target=delta_90 | split=valid | baseline=zero
[EVAL] L30 | split=valid | baseline=mean | DELTA
  -> L30 | target=delta_60 | split=valid | baseline=mean
  -> L30 | target=delta_90 | split=valid 

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,family
0,naive_mean,test,30,delta_60,60,53.496417,83.313665,-0.000008,0.527432,naive
1,naive_zero,test,30,delta_60,60,53.522681,83.313808,-0.000012,NaN,naive
2,naive_mean,valid,30,delta_60,60,35.266343,50.223012,-0.000549,0.546686,naive
3,naive_zero,valid,30,delta_60,60,35.311863,50.237993,-0.001146,NaN,naive
4,naive_mean,test,30,delta_90,90,67.232743,103.990683,-0.000039,0.527462,naive
...,...,...,...,...,...,...,...,...,...,...
75,naive_zero,valid,180,ret_60,60,0.002216,0.002996,-0.000887,NaN,naive
76,naive_mean,test,180,ret_90,90,0.003752,0.005949,-0.000424,0.524495,naive
77,naive_zero,test,180,ret_90,90,0.003757,0.005948,-0.000018,NaN,naive
78,naive_mean,valid,180,ret_90,90,0.002687,0.003698,-0.000014,0.548872,naive


### **11.2. Unir métricas Naive**

In [ ]:
df_naive_all_sizes

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,family
0,naive_mean,test,30,delta_60,60,53.496417,83.313665,-0.000008,0.527432,naive
1,naive_zero,test,30,delta_60,60,53.522681,83.313808,-0.000012,NaN,naive
2,naive_mean,valid,30,delta_60,60,35.266343,50.223012,-0.000549,0.546686,naive
3,naive_zero,valid,30,delta_60,60,35.311863,50.237993,-0.001146,NaN,naive
4,naive_mean,test,30,delta_90,90,67.232743,103.990683,-0.000039,0.527462,naive
...,...,...,...,...,...,...,...,...,...,...
75,naive_zero,valid,180,ret_60,60,0.002216,0.002996,-0.000887,NaN,naive
76,naive_mean,test,180,ret_90,90,0.003752,0.005949,-0.000424,0.524495,naive
77,naive_zero,test,180,ret_90,90,0.003757,0.005948,-0.000018,NaN,naive
78,naive_mean,valid,180,ret_90,90,0.002687,0.003698,-0.000014,0.548872,naive


### **11.3. Guardar métricas**

In [ ]:
save_seq2one_metrics(
    df_naive_all_sizes,
    name="naive",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/seq2one_naive_metrics.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/seq2one_naive_metrics.parquet')